# LAB 06 — Transformations on RetailHub Silver Data

## Scenario

RetailHub's Silver layer is in place — typed orders, customers, and products tables.
Your job is the modeling work on top of it: enrich orders with customer and product attributes, prove a broadcast join from the physical plan, remove the duplicates introduced by the overlapping stream extract, reshape data with arrays, and profile revenue with exact, approximate, and windowed aggregations.

This is the hands-on companion to the self-study demo `05a — Transformations & Modeling` and covers the largest exam domain (Data Transformation and Modeling — 22%).

## Key Concepts

### Join types and keys
`df.join(other, on="key", how="inner"|"left")` — passing the key as a string de-duplicates the join column. A chain of joins can use a **different key per step** (orders→customers on `customer_id`, orders→products on `product_id`).
`inner` drops rows without a match; `left` keeps every left-side row with `NULL`s for missing matches.

### Broadcast join
`large_df.join(broadcast(small_df), on="key")` ships the small table to every executor — **no shuffle of the large side**. Spark auto-broadcasts below `spark.sql.autoBroadcastJoinThreshold` (10 MB default). Verify in the physical plan: look for `BroadcastHashJoin` / `BroadcastExchange` in `df.explain()` output.

### Window-based deduplication
```python
w = Window.partitionBy("business_key").orderBy("preference_col")
df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
```
Unlike `dropDuplicates()`, you control **which** duplicate survives via `orderBy`.

### Arrays
`F.collect_set(col)` (distinct) / `F.collect_list(col)` build arrays in an aggregation; `F.split(str_col, sep)` builds them from strings; `F.explode(arr)` unpacks one row per element; `F.size(arr)` counts elements.

### Exact vs approximate distinct counts
`F.count_distinct(col)` is exact but shuffles all distinct values. `F.approx_count_distinct(col)` uses HyperLogLog++ — one pass, default error ±5%. `df.summary()` returns count/mean/stddev/min/percentiles/max in one call.

### Window functions
Ranking (`rank`, `dense_rank`, `row_number`) answers "which is the best per group"; offset functions (`lag`, `lead`) answer "how does this row compare to the previous one". Each question usually needs its **own** window spec with a different `orderBy`.

### Shuffle partitions
`spark.sql.shuffle.partitions` (default 200) sets the number of post-shuffle partitions. For small data, 200 partitions = 200 mostly-empty tasks. Always restore config changes after an experiment.

## Prerequisites

- Run the **Setup** and preparation cells first — they build the three Silver tables (`orders_silver_lab06`, `customers_silver_lab06`, `products_silver_lab06`) from the raw RetailHub files
- Orders are **intentionally dirty** — several tasks rely on this: ~9% have no matching customer (null `customer_id`, orphan `CUST999999` ids, customer rows without an id), and some `order_id`s are NULL or repeated
- `orders_stream_001.json` intentionally **replays the head of `orders_batch.json`** — Task 3 is built on this overlap

## Tasks Overview

| Task | Topic | What you will do |
|------|-------|-----------------|
| 1 | Joins | Chain inner and left joins across `customer_id` and `product_id`; compare row counts |
| 2 | Broadcast join | Force a broadcast, capture `df.explain()` output, find the Broadcast operator |
| 3 | Dedup | Union batch + stream_001, count duplicates, keep one row per `order_id` with `row_number()` |
| 4 | Arrays | Build per-customer baskets with `collect_set`, unpack with `explode` |
| 5 | Aggregations | Segment stats with `count` / `approx_count_distinct` / `avg`; `summary()` |
| 6 | Windows | `rank` months by revenue and `lag` month-over-month per segment |
| 7 | Challenge | Time one aggregation under two `spark.sql.shuffle.partitions` settings |

## Hints — Tasks 1–4

### Task 1: Multi-table joins
Chain two `.join()` calls — one per key. The join type is the third argument:
```python
df.join(other_df, on="<key>", how="<inner-or-left>")
```
Build the inner chain first, then copy it and switch `how=`. Finish with `.select(*cols)`.
Check: the left variant must have exactly as many rows as `orders_df`; the inner variant fewer. Ask yourself which rows vanished (~9%: null or orphan `customer_id`s and customers without an id).

---

### Task 2: Broadcast join
Two ingredients:
1. Wrap the **small** side: `orders_df.join(broadcast(products_df), on=..., how="inner")`
2. `df.explain()` prints instead of returning — capture stdout:
```python
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    <your_df>.explain()
plan_str = buf.getvalue()
```
Search `plan_str` for `Broadcast`. (`io` and `contextlib` are already imported in Configuration.)

---

### Task 3: Dedup with a window
Three steps:
1. Read the stream file with `spark.read.format("json")` (the missing `order_ts` / `order_month` columns are filled with NULLs by `allowMissingColumns=True`), then tag each side with `.withColumn("source", F.lit("batch"/"stream"))` and union with `unionByName(..., allowMissingColumns=True)`.
2. Duplicates = total rows − distinct `order_id`s. Expect about 15.8k — more than the ~9.4k replayed rows, because all NULL `order_id`s count as **one** key and some ids repeat inside the files. Deduplicating by `order_id` collapses the NULLs into one row, so in real pipelines filter `order_id IS NOT NULL` first.
3. Window dedup: partition by `order_id`, order by `source` (alphabetically `batch` < `stream` — so the batch row gets `rn = 1`), filter `rn = 1`, drop `rn`.

---

### Task 4: collect_set + explode
- Baskets: `groupBy("customer_id").agg(F.collect_set(...).alias("products"))`, then `.withColumn("basket_size", F.size("products"))`.
- Explode: `baskets.select("customer_id", F.explode("products").alias("product_id"))`.
Check: `printSchema()` should show `products: array<string>`; the exploded row count equals the number of **distinct** (customer, product) pairs — `collect_set` already deduplicated.

## Hints — Tasks 5–7

### Task 5: Aggregations
- `segment_stats`: one `groupBy("customer_segment").agg(...)` with three aggregate expressions — alias each one exactly as the validation expects (`order_count`, `approx_customers`, `avg_amount`).
- Exact vs approx: `orders_df.select(F.count_distinct("customer_id").alias("n")).first()["n"]` — same pattern with `approx_count_distinct`.
- `summary_df = orders_df.select("total_amount").summary()` — no arguments needed.

---

### Task 6: rank & lag
You need **two** window specs over the same monthly aggregate:
```python
w_rank = Window.partitionBy(...).orderBy(F.desc("revenue"))
w_time = Window.partitionBy(...).orderBy("order_month")
```
Then three `.withColumn()` calls: `revenue_rank` (rank over `w_rank`), `prev_revenue` (lag over `w_time`), `mom_change` (current − previous).
Filter out `NULL` segment/month **before** grouping — dirty rows would otherwise form their own group.

---

### Task 7: Tuning experiment
The skeleton gives you `timed_agg()`. Your work is config bookkeeping:
1. `spark.conf.get("spark.sql.shuffle.partitions")` → save it
2. `spark.conf.set(...)` to `"200"`, time it; set to `"8"`, time it
3. `spark.conf.set(...)` back to the saved value — the validation checks this!

Don't be surprised if the difference is small: AQE (on by default) already coalesces tiny shuffle partitions. The habit being trained: measure, and always restore.

## Summary

### What each task trains

| Skill | Where it appears on the exam |
|---|---|
| Join type selection | Scenario questions: "which rows are lost?" |
| `broadcast()` + threshold config | Performance questions on join strategies |
| Window dedup (`row_number` = 1) | "Keep the latest record per key" patterns |
| `explode` / arrays | Complex-type transformation questions |
| `approx_count_distinct` | Cost/accuracy trade-off questions |
| `rank` vs `dense_rank` vs `row_number`, `lag`/`lead` | Classic window-function questions |
| `spark.sql.shuffle.partitions` | One of the 4 named tuning configs |

### If you get stuck
1. Re-read the Hint cell after the task's code cell — the syntax pattern is always there.
2. Check the validation cell: the assert messages tell you exactly what is expected.
3. Full answers: `notebooks/solution/lab_06_transformations_solution.ipynb` (after the lab!).